# 02 · Números: divisibilidad, primos, MCD y mcm, aritmética modular

Tópico 02 del temario.

## Divisibilidad

`a` es divisible entre `b` si `a % b == 0`. Si `a = b · k`, decimos que `b` es **divisor** de `a` y `a` es **múltiplo** de `b`.

In [ ]:
assert 12 % 4 == 0 and 12 % 5 == 2

# Cuántos múltiplos de k hay entre 1 y n
n, k = 100, 7
assert n // k == 14

# Cuántos múltiplos de k hay entre a y b (inclusive)
a, b = 20, 50
assert b // k - (a - 1) // k == 5          # 21, 28, 35, 42, 49

### Divisores en O(√n)

Los divisores vienen en pares `(d, n // d)`. Basta probar hasta `√n`.

In [ ]:
def divisores(n):
    res = []
    d = 1
    while d * d <= n:
        if n % d == 0:
            res.append(d)
            if d != n // d:
                res.append(n // d)
        d += 1
    return sorted(res)


assert divisores(36) == [1, 2, 3, 4, 6, 9, 12, 18, 36]
assert divisores(13) == [1, 13]

## Primos

Un **primo** tiene exactamente dos divisores: 1 y él mismo (el 1 **no** es primo).

### Prueba de primalidad en O(√n)

In [ ]:
def es_primo(n):
    if n < 2:
        return False
    d = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += 1
    return True


assert [x for x in range(20) if es_primo(x)] == [2, 3, 5, 7, 11, 13, 17, 19]
assert es_primo(1_000_000_007)

### Criba de Eratóstenes: todos los primos hasta `n`

Marca los múltiplos de cada primo. `O(n log log n)`.

In [ ]:
def criba(n):
    es = [True] * (n + 1)
    es[0] = es[1] = False
    i = 2
    while i * i <= n:
        if es[i]:
            for j in range(i * i, n + 1, i):
                es[j] = False
        i += 1
    return es


def primos_hasta(n):
    es = criba(n)
    return [i for i in range(n + 1) if es[i]]


assert primos_hasta(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert len(primos_hasta(100000)) == 9592

### Factorización en primos

In [ ]:
def factorizar(n):
    factores = {}
    d = 2
    while d * d <= n:
        while n % d == 0:
            factores[d] = factores.get(d, 0) + 1
            n //= d
        d += 1
    if n > 1:                       # lo que queda es un primo
        factores[n] = factores.get(n, 0) + 1
    return factores


assert factorizar(360) == {2: 3, 3: 2, 5: 1}      # 2^3 * 3^2 * 5
assert factorizar(97) == {97: 1}

# Cantidad de divisores = producto de (exponente + 1)
def cantidad_divisores(n):
    total = 1
    for e in factorizar(n).values():
        total *= e + 1
    return total


assert cantidad_divisores(36) == 9

## MCD y mcm

- **MCD** (máximo común divisor): el mayor número que divide a ambos.
- **mcm** (mínimo común múltiplo): el menor número que ambos dividen.
- Relación: `mcd(a, b) · mcm(a, b) = a · b`.

### Algoritmo de Euclides

`mcd(a, b) = mcd(b, a % b)` hasta que `b` sea 0. Es muy rápido: `O(log n)`.

In [ ]:
import math


def mcd(a, b):
    while b:
        a, b = b, a % b
    return a


def mcm(a, b):
    return a // mcd(a, b) * b


assert mcd(12, 18) == 6 and mcm(4, 6) == 12
assert mcd(17, 5) == 1                         # coprimos
assert mcd(12, 18) == math.gcd(12, 18) and mcm(4, 6) == math.lcm(4, 6)

# Para una lista, aplica de a pares
def mcd_lista(arr):
    r = 0
    for x in arr:
        r = mcd(r, x)
    return r


def mcm_lista(arr):
    r = 1
    for x in arr:
        r = mcm(r, x)
    return r


assert mcd_lista([12, 18, 30]) == 6
assert mcm_lista([2, 3, 4, 5]) == 60

**Aplicación típica del mcm:** dos luces parpadean cada 4 y cada 6 segundos. ¿Cada cuántos segundos coinciden? `mcm(4, 6) = 12`.

## Aritmética modular

Cuando los números se vuelven enormes, el problema pide el resultado **módulo `m`** (el residuo de dividir entre `m`).

Propiedades (valen para suma, resta y multiplicación):

- `(a + b) % m == ((a % m) + (b % m)) % m`
- `(a - b) % m == ((a % m) - (b % m)) % m`
- `(a * b) % m == ((a % m) * (b % m)) % m`

⚠️ **La división no funciona así.** Para dividir se usa el inverso modular (más abajo).

In [ ]:
M = 10 ** 9 + 7

a, b = 10 ** 18, 10 ** 17 + 3
assert (a + b) % M == ((a % M) + (b % M)) % M
assert (a * b) % M == ((a % M) * (b % M)) % M
assert (3 - 10) % 7 == 0                  # Python ya devuelve un residuo no negativo

# Factorial módulo M sin crecer demasiado
def factorial_mod(n, m=M):
    r = 1
    for i in range(2, n + 1):
        r = r * i % m
    return r


assert factorial_mod(10) == 3628800
assert factorial_mod(20) == 2432902008176640000 % M

### Potencia modular rápida

`pow(a, b, m)` calcula `a^b mod m` en `O(log b)`. Úsala siempre que haya una potencia grande.

In [ ]:
assert pow(2, 10, 1000) == 24
assert pow(2, 10 ** 18, M) == pow(2, 10 ** 18 % (M - 1), M)      # (pequeño teorema de Fermat)

# Versión manual (por si el problema no permite pow con tres argumentos)
def potencia_mod(a, b, m):
    r = 1
    a %= m
    while b:
        if b & 1:
            r = r * a % m
        a = a * a % m
        b >>= 1
    return r


assert potencia_mod(3, 200, 1000) == pow(3, 200, 1000)

### Último dígito y ciclos

El último dígito de una potencia es un cálculo módulo 10.

In [ ]:
assert pow(7, 2024, 10) == 1
assert pow(2, 100, 10) == 6

### Inverso modular

Si `m` es primo y `a` no es múltiplo de `m`, el inverso de `a` es `a^(m-2) mod m`. En Python 3.8+ también: `pow(a, -1, m)`.

In [ ]:
inv2 = pow(2, -1, M)
assert 2 * inv2 % M == 1

# Dividir módulo M: a / b  ->  a * inverso(b)
def division_mod(a, b, m=M):
    return a * pow(b, -1, m) % m


assert division_mod(10, 2) == 5

### Dígitos y bases

In [ ]:
assert [int(c) for c in str(4729)] == [4, 7, 2, 9]
assert sum(int(c) for c in str(4729)) == 22

assert bin(10) == "0b1010" and int("1010", 2) == 10
assert int("ff", 16) == 255 and format(255, "x") == "ff"

## Problemas tipo (estilo grader)

**P1.** `contar_primos(a, b)`: cuántos primos hay en `[a, b]` (con `b ≤ 10^6`).
**P2.** `suma_divisores(n)`: suma de todos los divisores de `n`.
**P3.** `es_perfecto(n)`: `True` si `n` es igual a la suma de sus divisores propios (6, 28, 496...).
**P4.** `luces(a, b)`: cada cuántos segundos coinciden dos luces con periodos `a` y `b`.

In [ ]:
def contar_primos(a, b):
    es = criba(b)
    return sum(1 for i in range(a, b + 1) if es[i])


def suma_divisores(n):
    return sum(divisores(n))


def es_perfecto(n):
    return n > 1 and suma_divisores(n) - n == n


def luces(a, b):
    return mcm(a, b)


assert contar_primos(10, 30) == 6          # 11 13 17 19 23 29
assert suma_divisores(12) == 28
assert [x for x in range(1, 500) if es_perfecto(x)] == [6, 28, 496]
assert luces(4, 6) == 12
print("ok")

## Para practicar

1. `es_potencia_de_dos(n)` (pista: `n & (n - 1)`).
2. `numeros_coprimos_con(n)`: cuántos `k` entre 1 y `n` cumplen `mcd(k, n) == 1`.
3. `primos_gemelos(n)`: pares de primos que difieren en 2 hasta `n`.
4. `ultimo_digito_factorial(n)` y `ceros_al_final_de_factorial(n)` (cuenta los factores de 5).
5. `suma_modular(arr, m)`: suma de todos los elementos módulo `m`.

## Errores típicos

- Tratar el `1` como primo.
- Probar divisores hasta `n` en lugar de hasta `√n` (TLE con `n ≈ 10^12`).
- Calcular `a ** b % m` con exponentes grandes: usa `pow(a, b, m)`.
- Olvidar el `% m` en cada multiplicación dentro de un bucle.
- Usar `/` en lugar de `//` y obtener un decimal (o perder precisión en números enormes).